# DSNN: what the phase does, and where it can do it

Supplementary material for our TDL Challenge 2026 **Track 1** submission,
which implements the **Directed Sheaf Neural Network** of *Sheaves Reloaded: A
Directional Awakening* (ICLR 2026, [arXiv:2506.02842](https://arxiv.org/abs/2506.02842)).

DSNN's whole contribution is one factor. It takes the sheaf Laplacian of Neural
Sheaf Diffusion and multiplies each off-diagonal block by a unit-modulus phase
read off the edge directions,

$$T^{(q)}_{uv} = \exp\!\big(i\,2\pi q\,(A_{uv} - A_{vu})\big), \qquad
  L^{\tilde{\mathcal{F}}} = L^{\mathcal{F}} \odot T^{(q)},$$

leaving the diagonal real. That is enough to make sheaf diffusion sensitive to
direction for the first time.

It is also enough to make the contribution invisible on every graph benchmark
in TopoBench, because all of them are undirected, and on an undirected graph
$A = A^{\top}$ so $T^{(q)} = 1$ for every $q$. The official evaluation notebook
structurally cannot show what this model does. This one does, and it doubles as
the usage guide for what we implemented.

Three things are established here:

1. **The paper's Figure 2 is reproduced at its exact scale.** On the directed
   stochastic block model of Appendix E ($n = 2500$, $C = 5$), where the label
   is recoverable *only* from edge directions, DSNN reaches 0.84 to 0.96 as
   the communities separate, while the same model with the phase switched off
   stays at 0.18 to 0.20, i.e. chance. The paper reports 86-96% against ~20%.
2. **On undirected input the charge $q$ is provably inert**, and we measure it:
   in float64 the imaginary part is *identically* zero and the spectrum does
   not move by a single bit across the whole $q$ grid. We also confirm
   empirically that every GraphUniverse graph is undirected.
3. **Theorems 1-5 hold numerically**, including the two that identify this
   operator with the Magnetic and Sign-Magnetic Laplacians. Along the way we
   settle a sign discrepancy in the paper's Definition 1 by showing it makes
   no difference to the model.

Sections 0 and 1 are the usage and feature tour; 2-6 are the verification;
7 is the reproduction; 8 asks whether an induced orientation changes what the
model *learns*.

In [ ]:
import os
import time
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings("ignore")

# Resolve the repository root whether the kernel starts in 2026_tdl_challenge/
# or at the top of the repo.
_HERE = Path.cwd()
REPO = _HERE if (_HERE / "topobench").is_dir() else _HERE.parent
assert (REPO / "topobench").is_dir(), f"could not locate the repo from {_HERE}"
os.environ.setdefault("PROJECT_ROOT", str(REPO))
MEDIA = REPO / "2026_tdl_challenge" / "media"
MEDIA.mkdir(parents=True, exist_ok=True)

import sys

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from topobench.nn.backbones.graph.dsnn import DSNNEncoder
from topobench.nn.backbones.graph.dsnn_utils import (
    complex_ops,
    laplace,
    laplacian_builders,
    orthogonal,
    phase,
)

# Every exactness claim below is measured in float64.
DTYPE = torch.float64
torch.manual_seed(0)

Q_GRID = (0.0, 0.1, 0.15, 0.2, 0.25, 0.5, 0.75, 1.0)
FAMILIES = ("diag", "bundle", "general")

# ----------------------------------------------------------------- figure style
# Two-series palette chosen for a >=3.0 CIEDE2000 separation under deuteranopia
# and protanopia simulation, so the figures survive both grayscale and the
# common colour-vision deficiencies.
SURFACE, INK, MUTED, GRID, AXIS = (
    "#ffffff",
    "#1b1f24",
    "#6b7580",
    "#e3e7ec",
    "#39414a",
)
SERIES = ("#2a78d6", "#eb6834", "#1a9e77", "#8d5bd0")
SIGNED = LinearSegmentedColormap.from_list(
    "signed", ["#2a78d6", "#f2f4f7", "#eb6834"]
)

plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": AXIS,
        "axes.labelcolor": INK,
        "axes.titlecolor": INK,
        "axes.titlesize": 11,
        "axes.titleweight": "600",
        "axes.labelsize": 9.5,
        "axes.grid": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "grid.color": GRID,
        "grid.linewidth": 0.7,
        "text.color": INK,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "xtick.labelsize": 8.5,
        "ytick.labelsize": 8.5,
        "legend.frameon": False,
        "legend.fontsize": 8.5,
        "figure.dpi": 110,
        "savefig.bbox": "tight",
    }
)


def save(fig, name):
    """Write a figure into 2026_tdl_challenge/media/."""
    path = MEDIA / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight", facecolor=SURFACE)
    print(f"wrote {path.relative_to(REPO)}")


print(f"repo   : {REPO}")
print(f"torch  : {torch.__version__}")

## 0. Using DSNN in TopoBench

Two entry points. The Hydra configs are what the benchmark uses; the raw
backbone is what the rest of this notebook uses, because it lets us look at the
operator directly.

We ship four configs. They differ in as few lines as possible so that any
comparison between them isolates one choice:

| config | family | `orientation` | why it exists |
|---|---|---|---|
| `graph/dsnn` | diagonal | `none` | the faithful model; most of the paper's Table 1 |
| `graph/dsnn_ortho` | orthogonal, $O(d)$ | `none` | the `O(d)-DSNN` row |
| `graph/dsnn_general` | general $d \times d$ | `none` | the `Gen-DSNN` row |
| `graph/dsnn_degree` | diagonal | `degree` | the only one whose complex path is live on undirected data |

In [ ]:
import hydra

for name in ("dsnn", "dsnn_ortho", "dsnn_general", "dsnn_degree"):
    hydra.core.global_hydra.GlobalHydra.instance().clear()
    with hydra.initialize(
        version_base="1.3", config_path=str(Path("..") / "configs")
    ):
        cfg = hydra.compose(
            config_name="run.yaml",
            overrides=[
                f"model=graph/{name}",
                "dataset=graph/graphuniverse_inductive",
            ],
        )
    backbone = hydra.utils.instantiate(cfg.model.backbone)
    transforms = cfg.get("transforms", None)
    print(
        f"{name:13s} {type(backbone).__name__}"
        f"  family={backbone.sheaf_type:8s} d={backbone.d}"
        f"  q={backbone.q}  orientation={backbone.orientation!r}"
        f"  in_channels={list(cfg.model.feature_encoder.in_channels)}"
        f"  transforms={list(transforms.keys()) if transforms else None}"
    )

print()
print(
    "transforms is None because dataset and model are both graph-domain, so no"
    "\nlifting is applied and no positional encoding is injected: DSNN needs"
    "\nneither. in_channels stays at the dataset's own 15 features."
)

In [ ]:
# The raw backbone, with the shape at every stage of Eq. 8 printed. This is
# the clearest way to see the real lifting of Appendix D: the cochain is
# [n*d, f] complex, carried as [2*n*d, f] real with Re stacked above Im.
graph = torch.tensor([[0, 1, 1, 2, 2, 0, 3, 4], [1, 0, 2, 1, 0, 2, 4, 3]])
num_nodes, in_dim, hidden, d = 5, 12, 24, 2
features = torch.randn(num_nodes, in_dim)

model = DSNNEncoder(
    input_dim=in_dim, hidden_dim=hidden, num_layers=2, d=d, q=0.25
).eval()
stack = model.get_sheaf_model()

print(f"input features                     {tuple(features.shape)}")
print(f"  after lin1 -> hidden_dim         ({num_nodes}, {stack.hidden_dim})")
print(
    f"  reshaped to the 0-cochain        "
    f"({num_nodes * d}, {stack.hidden_channels})   [n*d, f], complex"
)
print(
    f"  carried in the real lifting      "
    f"({2 * num_nodes * d}, {stack.hidden_channels})   [Re ; Im]"
)
print(
    f"  unwind -> (Re || Im)             "
    f"({num_nodes}, {2 * stack.hidden_dim})"
)
print(
    f"  after lin2                       {tuple(model(features, graph).shape)}"
)
print()
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"repr      : {model}")

Equivalently, from the command line:

```bash
python -m topobench model=graph/dsnn         dataset=graph/MUTAG
python -m topobench model=graph/dsnn_ortho   dataset=graph/MUTAG
python -m topobench model=graph/dsnn_general dataset=graph/MUTAG
python -m topobench model=graph/dsnn_degree  dataset=graph/MUTAG
```

## 1. What every knob does

The backbone exposes seven choices beyond the usual width and depth. This
section demonstrates each one on the same small graph, so the panels are
comparable.

**Claim.** The imaginary part of the operator is exactly zero for every
`orientation="none"` cell regardless of $q$, and structured elsewhere. The
three families differ in how many parameters they predict per edge and how
dense the resulting blocks are, but not in whether the phase applies.

In [ ]:
def build_operator(
    edge_index,
    num_nodes,
    q,
    family="diag",
    d=2,
    orientation="none",
    phase_sign=1,
    seed=0,
    **kwargs,
):
    """Assemble the dense complex operator for a given configuration."""
    support = laplace.symmetrize_support(edge_index, num_nodes)
    left_right, pair = laplace.compute_left_right_map_index(support, num_nodes)
    sign = phase.pair_sign(edge_index, support, pair, num_nodes, orientation)
    cos, sin = phase.phase_from_sign(sign, q, phase_sign, dtype=DTYPE)
    generator = torch.Generator().manual_seed(seed)
    if family == "diag":
        maps = torch.randn(
            support.size(1), d, generator=generator, dtype=DTYPE
        )
    elif family == "bundle":
        maps = torch.randn(
            support.size(1),
            orthogonal.num_orthogonal_params(d),
            generator=generator,
            dtype=DTYPE,
        )
    else:
        maps = torch.randn(
            support.size(1), d, d, generator=generator, dtype=DTYPE
        )
    builder = laplacian_builders.LAPLACIAN_BUILDERS[family](
        num_nodes, support, d, left_right, pair, cos, sin, **kwargs
    )
    real_i, real_v, imag_i, imag_v, _ = builder.hermitian_parts(maps)
    dense = complex_ops.hermitian_to_dense(
        real_i, real_v, imag_i, imag_v, num_nodes * d
    )
    return dense, builder, sign


# A degree-heterogeneous graph, so an induced degree orientation is non-trivial.
TOUR = torch.tensor(
    [
        [0, 1, 0, 2, 0, 3, 1, 2, 3, 4, 0, 5, 5, 6, 4, 6],
        [1, 0, 2, 0, 3, 0, 2, 1, 4, 3, 5, 0, 6, 5, 6, 4],
    ]
)
TOUR_N = 7

rows = ("none", "degree", "index")
cols = (0.0, 0.1, 0.25, 0.5)
fig, axes = plt.subplots(
    len(rows), len(cols), figsize=(8.6, 6.4), constrained_layout=True
)
limit = 0.0
panels = {}
for r, orient in enumerate(rows):
    for c, q in enumerate(cols):
        dense, _, sign = build_operator(TOUR, TOUR_N, q, orientation=orient)
        panels[(r, c)] = (dense.imag.numpy(), sign)
        limit = max(limit, float(np.abs(dense.imag.numpy()).max()))
limit = limit or 1.0
for (r, c), (imag, sign) in panels.items():
    ax = axes[r, c]
    ax.imshow(imag, cmap=SIGNED, vmin=-limit, vmax=limit)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    if r == 0:
        ax.set_title(f"q = {cols[c]}", fontsize=10)
    if c == 0:
        oriented = int((sign != 0).sum())
        ax.set_ylabel(
            f"{rows[r]}\n{oriented}/{sign.numel()} oriented", fontsize=9
        )
fig.suptitle(
    "Imaginary part of the operator: orientation (rows) against charge "
    "(columns)",
    fontsize=11,
)
save(fig, "feature_tour_operator")
plt.show()

> **Expected output** &mdash; `media/feature_tour_operator.png`
>
> ![Grid of heatmaps of the imaginary part of the operator, one row per orientation and one column per charge. Every cell of the none row is flat white because the imaginary part is exactly zero, while the degree and index rows show structure that rotates with q.](media/feature_tour_operator.png)

In [ ]:
# Cost of the three families. The number of predicted parameters per arc and
# the non-zeros of the assembled operator are what the stalk dimension buys.
print(
    f"{'family':9s} {'d':>2}  {'params/arc':>10}  {'nnz(L)':>8}  {'nnz lifted':>10}"
)
for family in FAMILIES:
    for d in (2, 3, 4):
        if family == "bundle" and d < 2:
            continue
        dense, builder, _ = build_operator(
            TOUR, TOUR_N, 0.25, family=family, d=d, orientation="index"
        )
        per_arc = {
            "diag": d,
            "bundle": orthogonal.num_orthogonal_params(d),
            "general": d * d,
        }[family]
        nnz = int((dense.abs() > 1e-12).sum())
        print(
            f"{family:9s} {d:>2}  {per_arc:>10}  {nnz:>8}  "
            f"{builder.lifted_index.size(1):>10}"
        )

fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.4), constrained_layout=True)
dims = [1, 2, 3, 4, 5]
for k, family in enumerate(FAMILIES):
    counts = [
        {
            "diag": d,
            "bundle": orthogonal.num_orthogonal_params(d),
            "general": d * d,
        }[family]
        for d in dims
    ]
    axes[0].plot(
        dims, counts, marker="o", color=SERIES[k], label=family, lw=1.8
    )
axes[0].set_xlabel("stalk dimension $d$")
axes[0].set_ylabel("predicted parameters per arc")
axes[0].set_title("Sheaf learner output width")
axes[0].set_xticks(dims)
axes[0].legend()

for k, family in enumerate(FAMILIES):
    widths = []
    for d in dims:
        if family == "bundle" and d < 2:
            widths.append(np.nan)
            continue
        _, builder, _ = build_operator(
            TOUR, TOUR_N, 0.25, family=family, d=d, orientation="index"
        )
        widths.append(builder.lifted_index.size(1))
    axes[1].plot(
        dims, widths, marker="o", color=SERIES[k], label=family, lw=1.8
    )
axes[1].set_xlabel("stalk dimension $d$")
axes[1].set_ylabel("non-zeros of the lifted operator")
axes[1].set_title("Operator size (real lifting)")
axes[1].set_xticks(dims)
axes[1].legend()
save(fig, "feature_tour_cost")
plt.show()

> **Expected output** &mdash; `media/feature_tour_cost.png`
>
> ![Two panels comparing the three restriction-map families: parameters predicted per arc by the sheaf learner, and non-zeros of the lifted operator, both against the stalk dimension.](media/feature_tour_cost.png)

In [ ]:
# The remaining knobs, each shown by what it changes about the operator.
base = dict(edge_index=TOUR, num_nodes=TOUR_N, q=0.25, orientation="index")

exact, _, _ = build_operator(**base, family="general", d=3, block_norm=True)
jacobi, _, _ = build_operator(**base, family="general", d=3, block_norm=False)
plain, _, _ = build_operator(**base, normalised=False)
shifted, _, _ = build_operator(**base, degree_shift=1.0)
plus, _, _ = build_operator(**base, phase_sign=1)
minus, _, _ = build_operator(**base, phase_sign=-1)

print(
    "normalised=False  ->  lambda_max = "
    f"{torch.linalg.eigvalsh(plain).max():.3f}  (unbounded, Eq. 5 skipped)"
)
print(
    "normalised=True   ->  lambda_max = "
    f"{torch.linalg.eigvalsh(exact).max():.6f}  (Thm 2 bound)"
)
print(
    "degree_shift=1.0  ->  lambda_max = "
    f"{torch.linalg.eigvalsh(shifted).max():.6f}  "
    "(the (D+I)^-1/2 of the reference implementation)"
)
print()
print("block_norm on the general family, whose degree blocks are full:")
print(
    f"  exact  D^-1/2 (eigh)      lambda_max = "
    f"{torch.linalg.eigvalsh(exact).max():.6f}   <= 2"
)
print(
    f"  Jacobi (block diagonal)   lambda_max = "
    f"{torch.linalg.eigvalsh(jacobi).max():.6f}   > 2, so Thm 2 fails"
)
print(
    "  Both are congruences, so both stay Hermitian and PSD; only the exact"
    "\n  one satisfies the bound. That is why block_norm defaults to True."
)
print()
print("phase_sign, the two readings of Definition 1:")
print(
    f"  conjugate operators   {torch.allclose(minus, plus.conj(), atol=1e-12)}"
)
print(
    f"  isospectral           "
    f"{torch.allclose(torch.linalg.eigvalsh(plus), torch.linalg.eigvalsh(minus), atol=1e-10)}"
)

**Read.** Every `none` panel is flat white: the imaginary part is not small,
it is exactly zero, and no value of $q$ changes that. Along the `degree` row
the phase switches on and its pattern rotates with $q$; at $q = 0.5$ the phase
is real again ($e^{\pm i\pi} = -1$), which is why that panel loses its
imaginary part too.

`degree` orients 14 of 16 edges here and `index` all 16. That gap is the price
of permutation equivariance: `degree` compares nodes only by isomorphism
invariants (degree, then the neighbourhood degree sum) and leaves genuine ties
undirected, whereas `index` breaks every tie using node identity and so is not
equivariant. On a degree-regular graph `degree` orients nothing at all and
coincides with `none`; we check that explicitly in section 8.

Only the general family has full degree blocks, and approximating $\tilde{D}^{-1/2}$ by their diagonal keeps the
operator positive semidefinite but pushes $\lambda_{\max}$ above 2, breaking
Theorem 2. The exact computation lands on 2.000000.

## 2. Theorems 1 and 2: this is a valid diffusion operator

**Claim.** For random directed graphs, random restriction maps, all three
families and every $q$ in the paper's grid: $\|L - L^{*}\|_\infty$ is at
machine epsilon, the diagonal is *exactly* real, and after Eq. 5 the spectrum
lies in $[0, 2]$ with the upper end attained.

In [ ]:
MIXED = torch.tensor(
    [[0, 1, 1, 2, 2, 0, 3, 4, 5], [1, 0, 2, 1, 0, 2, 4, 3, 3]]
)
MIXED_N = 7

print(
    f"{'family':9s} {'q':>5}  {'max|L-L*|':>11}  {'max|Im diag|':>12}"
    f"  {'lambda_min':>11}  {'lambda_max':>10}"
)
records = {}
for family in FAMILIES:
    for q in Q_GRID:
        dense, _, _ = build_operator(
            MIXED, MIXED_N, q, family=family, d=3, orientation="index"
        )
        herm = float((dense - dense.conj().T).abs().max())
        diag_imag = float(dense.diagonal().imag.abs().max())
        ev = torch.linalg.eigvalsh(dense)
        records.setdefault(family, []).append(
            (q, herm, float(ev.min()), float(ev.max()))
        )
        print(
            f"{family:9s} {q:>5}  {herm:>11.2e}  {diag_imag:>12.2e}"
            f"  {float(ev.min()):>+11.2e}  {float(ev.max()):>10.6f}"
        )
print()
print(f"float64 epsilon = {np.finfo(np.float64).eps:.2e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.6), constrained_layout=True)

for k, family in enumerate(FAMILIES):
    qs = [r[0] for r in records[family]]
    herm = [max(r[1], 1e-18) for r in records[family]]
    axes[0].semilogy(
        qs, herm, marker="o", color=SERIES[k], label=family, lw=1.6
    )
axes[0].axhline(
    np.finfo(np.float64).eps,
    color=MUTED,
    ls="--",
    lw=1.0,
    label="float64 eps",
)
axes[0].set_xlabel("charge $q$")
axes[0].set_ylabel(r"$\max|L - L^{*}|$")
axes[0].set_title("Theorem 1: Hermitian to machine precision")
axes[0].legend()

# Spectra of the normalized operator, overplotted across q.
for k, family in enumerate(FAMILIES):
    for j, q in enumerate(Q_GRID):
        dense, _, _ = build_operator(
            MIXED, MIXED_N, q, family=family, d=3, orientation="index"
        )
        ev = torch.linalg.eigvalsh(dense).numpy()
        axes[1].plot(
            np.arange(ev.size),
            ev,
            color=SERIES[k],
            alpha=0.55,
            lw=1.0,
            label=family if j == 0 else None,
        )
axes[1].axhline(2.0, color=INK, ls="--", lw=1.2)
axes[1].annotate(
    r"$\lambda = 2$ (Thm 2)",
    xy=(0.02, 2.0),
    xytext=(0.02, 1.78),
    xycoords=("axes fraction", "data"),
    color=INK,
    fontsize=8.5,
)
axes[1].set_xlabel("eigenvalue index")
axes[1].set_ylabel(r"$\lambda$ of $L_N$")
axes[1].set_title("Theorem 2: spectrum inside $[0, 2]$")
axes[1].legend()
save(fig, "hermitian_and_spectrum")
plt.show()

> **Expected output** &mdash; `media/hermitian_and_spectrum.png`
>
> ![Left, the Hermitian residual of the operator against the charge, sitting at machine epsilon for all three families. Right, the spectrum of the normalized operator, contained in the interval zero to two with the upper end attained.](media/hermitian_and_spectrum.png)

**Read.** Hermitian-ness sits at $10^{-16}$, i.e. one or two units in the
last place, uniformly in $q$ and in the family. The diagonal's imaginary
part is not merely small but bit-zero: the phase is applied only to the
off-diagonal blocks, never to the degree blocks. Every normalized spectrum
stops at exactly 2.0, so the bound of Theorem 2 is tight.

## 3. Theorem 3: on undirected input, $q$ does nothing

This is the section that decides what the challenge benchmark can and cannot
measure.

**Claim.** Three things, each exact rather than approximate. On an undirected
graph (i) the imaginary part of the operator is identically zero for every $q$;
(ii) the operator is *bitwise* identical across the whole $q$ grid; and (iii) it
equals the real sheaf Laplacian that TopoBench's Neural Sheaf Diffusion port
builds from the same restriction maps. And: every GraphUniverse graph is
undirected, so this is the regime the benchmark runs in.

In [ ]:
UNDIRECTED = torch.tensor([[0, 1, 1, 2, 2, 0, 3, 4], [1, 0, 2, 1, 0, 2, 4, 3]])

print("(i) and (ii) -- the operator itself")
for family in FAMILIES:
    base, _, _ = build_operator(UNDIRECTED, MIXED_N, 0.0, family=family, d=3)
    imag_zero, identical, drift = True, True, 0.0
    for q in Q_GRID:
        dense, _, _ = build_operator(
            UNDIRECTED, MIXED_N, q, family=family, d=3
        )
        imag_zero &= bool(
            torch.equal(dense.imag, torch.zeros_like(dense.imag))
        )
        identical &= bool(torch.equal(dense, base))
        drift = max(
            drift,
            float(
                (torch.linalg.eigvalsh(dense) - torch.linalg.eigvalsh(base))
                .abs()
                .max()
            ),
        )
    print(
        f"  {family:9s} Im(L) bit-zero for all q: {imag_zero}"
        f"   L bitwise identical for all q: {identical}"
        f"   max spectral drift: {drift:.1e}"
    )

print()
print("(iii) -- equality with the real sheaf Laplacian of the NSD port")
from topobench.nn.backbones.graph.nsd_utils import (
    laplacian_builders as nsd_builders,
)

for family, reference_class in (
    ("diag", nsd_builders.DiagLaplacianBuilder),
    ("general", nsd_builders.GeneralLaplacianBuilder),
):
    d = 3
    support = laplace.symmetrize_support(UNDIRECTED, MIXED_N)
    generator = torch.Generator().manual_seed(0)
    maps = (
        torch.randn(support.size(1), d, generator=generator, dtype=DTYPE)
        if family == "diag"
        else torch.randn(
            support.size(1), d, d, generator=generator, dtype=DTYPE
        )
    )
    left_right, pair = laplace.compute_left_right_map_index(support, MIXED_N)
    sign = phase.pair_sign(UNDIRECTED, support, pair, MIXED_N, "none")
    cos, sin = phase.phase_from_sign(sign, 0.37, 1, dtype=DTYPE)
    ours = laplacian_builders.LAPLACIAN_BUILDERS[family](
        MIXED_N, support, d, left_right, pair, cos, sin, normalised=False
    )
    real_i, real_v, imag_i, imag_v, _ = ours.hermitian_parts(maps)
    mine = complex_ops.hermitian_to_dense(
        real_i, real_v, imag_i, imag_v, MIXED_N * d
    )
    (index, value), _ = reference_class(MIXED_N, support, d=d)(maps)
    theirs = torch.zeros(MIXED_N * d, MIXED_N * d, dtype=DTYPE)
    theirs.index_put_((index[0], index[1]), value, accumulate=True)
    print(
        f"  {family:9s} identical to nsd_utils at q=0.37: "
        f"{torch.allclose(mine.real, theirs, atol=1e-14)}"
        f"   (max abs difference {float((mine.real - theirs).abs().max()):.1e})"
    )
print(
    "  The orthogonal family is excluded only because our parameterization\n"
    "  takes d(d-1)/2 inputs instead of d(d+1)/2, so the two cannot be fed\n"
    "  identical parameters; it is covered against a dense delta* delta."
)

In [ ]:
# Is GraphUniverse really undirected? Generate a small family and check,
# rather than taking it on trust.
from torch_geometric.utils import contains_self_loops, is_undirected

PROBE = {
    "universe_parameters": {
        "K": 20,
        "feature_dim": 15,
        "center_variance": 0.2,
        "cluster_variance": 0.4,
        "edge_propensity_variance": 0.5,
        "seed": 42,
    },
    "family_parameters": {
        "n_graphs": 12,
        "n_nodes_range": [50, 120],
        "n_communities_range": [5, 10],
        "degree_separation_range": [0.5, 1.0],
        "seed": 42,
    },
    "task": "community_detection",
}

try:
    from graph_universe import GraphUniverseDataset

    total = undirected = loops = 0
    for label, homophily, degree, power_law in (
        ("h_lo__d_lo", [0.0, 0.1], [1.0, 2.5], [1.5, 2.0]),
        ("h_mid__d_hi", [0.4, 0.6], [4.0, 5.0], [1.5, 2.0]),
        ("h_hi__d_hi", [0.9, 1.0], [4.0, 5.0], [4.0, 5.0]),
    ):
        params = {
            "universe_parameters": dict(PROBE["universe_parameters"]),
            "family_parameters": dict(
                PROBE["family_parameters"],
                homophily_range=homophily,
                avg_degree_range=degree,
                power_law_exponent_range=power_law,
            ),
            "task": "community_detection",
        }
        dataset = GraphUniverseDataset(
            root=str(REPO / "datasets" / f"_nb_probe_{label}"),
            parameters=params,
        )
        for item in dataset:
            total += 1
            undirected += int(is_undirected(item.edge_index))
            loops += int(contains_self_loops(item.edge_index))
    print(f"sampled {total} GraphUniverse graphs across 3 grid corners")
    print(f"  undirected      : {undirected}/{total}")
    print(f"  with self-loops : {loops}/{total}")
    print()
    print(
        "So on the challenge grid A = A^T, the phase is 1, and graph/dsnn is a"
        "\nreal-valued model no matter what q is set to."
    )
except Exception as exc:  # pragma: no cover - depends on the optional package
    print(f"could not generate a probe family ({exc})")
    print(
        "graph_universe builds every sample with networkx.Graph and calls"
        "\nto_undirected on the edge index (graph_sample.py), so the graphs are"
        "\nundirected by construction."
    )

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.0, 3.5), constrained_layout=True)

# All eight spectra coincide, so the eight lines overplot exactly. One visible
# curve is the result, not a rendering accident -- hence the annotation.
for j, q in enumerate(Q_GRID):
    dense, _, _ = build_operator(UNDIRECTED, MIXED_N, q, family="diag", d=3)
    ev = torch.linalg.eigvalsh(dense).numpy()
    axes[0].plot(
        np.arange(ev.size),
        ev,
        color=SERIES[0],
        lw=2.6 - 0.25 * j,
        alpha=0.9,
        label=f"q = {q}" if j in (0, 7) else None,
    )
axes[0].set_xlabel("eigenvalue index")
axes[0].set_ylabel(r"$\lambda$ of $L_N$")
axes[0].set_title("Undirected input: all eight $q$ overplot exactly")
axes[0].legend()

# The deviation across q is a single number, so state it rather than plotting
# eight indistinguishable zeros on a log axis.
labels, drifts = [], []
for family in FAMILIES:
    base, _, _ = build_operator(UNDIRECTED, MIXED_N, 0.0, family=family, d=3)
    worst = 0.0
    for q in Q_GRID:
        dense, _, _ = build_operator(
            UNDIRECTED, MIXED_N, q, family=family, d=3
        )
        worst = max(worst, float((dense - base).abs().max()))
    labels.append(family)
    drifts.append(worst)
axes[1].bar(
    labels, [max(v, 1e-20) for v in drifts], color=SERIES[0], width=0.5
)
axes[1].set_yscale("log")
axes[1].set_ylim(1e-20, 1e-14)
axes[1].axhline(np.finfo(np.float64).eps, color=MUTED, ls="--", lw=1.0)
axes[1].set_ylabel(r"$\max_q |L(q) - L(0)|$")
axes[1].set_title("Deviation across the whole $q$ grid")
for x, value in enumerate(drifts):
    axes[1].annotate(
        "exactly 0" if value == 0 else f"{value:.0e}",
        xy=(x, 1e-19),
        ha="center",
        fontsize=9,
        color=INK,
    )
save(fig, "q_inertness")
plt.show()

> **Expected output** &mdash; `media/q_inertness.png`
>
> ![Left, the sorted spectrum of the normalized operator on undirected input with all eight charges overplotted and indistinguishable. Right, the deviation from the q equals zero operator across the whole grid, which is exactly zero rather than merely small.](media/q_inertness.png)

**Read.** The deviation is not $10^{-16}$, it is zero: the same floating
point numbers come out for every charge, because every phase is the literal
constant 1. Combined with the census, this settles what the benchmark can show.

Two consequences follow:

* `graph/dsnn` on GraphUniverse *is* Neural Sheaf Diffusion with the readout of
  Eq. 8. Its leaderboard number should be read that way, and we say so in the
  pull request.
* `graph/dsnn_degree` exists so the complex path is reachable at all. It is a
  deviation from the paper, labelled as one.

## 4. Theorem 4, and a sign typo in Definition 1

For a *Trivial* Directed Cellular Sheaf ($d = 1$, one restriction map fixed to
1 and the other to $T^{(q)}_{uv}$) the operator should become the **Magnetic
Laplacian** of MagNet, and at $q = 1/4$ the **Sign-Magnetic Laplacian** of
SigMaNet. So DSNN contains both as special cases.

**Claim.** On a digon-free directed graph our operator equals $2L^{(q)}$; the
factor 2 is the paper's own remark that a one-way arc has $A_{s,uv} = 1/2$, and
it disappears on an all-digon graph. At $q = 1/4$, $L^{(1/4)} = L^{\sigma}$.

**And a discrepancy.** Definition 1 prints $\exp(+i2\pi q(\cdot))$, but its
worked example ($q = 1/4$ giving $-i$) needs the conjugate. We show below that
*either* reading satisfies Theorem 4 against the matching MagNet convention,
and that the two operators are conjugate and isospectral, so the typo has no
consequence for the model.

In [ ]:
def binary_adjacency(edge_index, num_nodes):
    """Dense binary adjacency of a directed graph."""
    adjacency = torch.zeros(num_nodes, num_nodes, dtype=DTYPE)
    adjacency[edge_index[0], edge_index[1]] = 1.0
    return adjacency


def magnetic_laplacian(edge_index, num_nodes, q, sign=1):
    """MagNet's L^(q) = D_s - A_s (*) exp(i 2 pi q (A - A^T))."""
    adjacency = binary_adjacency(edge_index, num_nodes)
    symmetric = (adjacency + adjacency.T) / 2
    theta = sign * 2 * torch.pi * q * (adjacency - adjacency.T)
    hermitian = symmetric.to(torch.complex128) * torch.complex(
        torch.cos(theta), torch.sin(theta)
    )
    return torch.diag(symmetric.sum(1)).to(torch.complex128) - hermitian


def sign_magnetic_laplacian(edge_index, num_nodes):
    """L^sigma, written from its definition rather than through exp()."""
    adjacency = binary_adjacency(edge_index, num_nodes)
    symmetric = (adjacency + adjacency.T) / 2
    ones = torch.ones(num_nodes, num_nodes, dtype=DTYPE)
    real = ones - torch.sign((adjacency - adjacency.T).abs())
    imag = torch.sign(adjacency.abs() - adjacency.T.abs())
    hermitian = symmetric.to(torch.complex128) * torch.complex(real, imag)
    return torch.diag(symmetric.abs().sum(1)).to(torch.complex128) - hermitian


def trivial_sheaf(edge_index, num_nodes, q, phase_sign=1):
    """Our operator with d = 1 and both base restriction maps equal to 1."""
    support = laplace.symmetrize_support(edge_index, num_nodes)
    left_right, pair = laplace.compute_left_right_map_index(support, num_nodes)
    sign = phase.pair_sign(edge_index, support, pair, num_nodes, "none")
    cos, sin = phase.phase_from_sign(sign, q, phase_sign, dtype=DTYPE)
    builder = laplacian_builders.DirectedDiagLaplacianBuilder(
        num_nodes, support, 1, left_right, pair, cos, sin, normalised=False
    )
    real_i, real_v, imag_i, imag_v, _ = builder.hermitian_parts(
        torch.ones(support.size(1), 1, dtype=DTYPE)
    )
    return complex_ops.hermitian_to_dense(
        real_i, real_v, imag_i, imag_v, num_nodes
    )


DIGON_FREE = torch.tensor([[0, 1, 2, 3, 0], [1, 2, 3, 0, 2]])
ALL_DIGON = torch.tensor([[0, 1, 1, 2, 2, 3, 3, 0], [1, 0, 2, 1, 3, 2, 0, 3]])
SMALL_N = 5

print("digon-free directed graph (A_s = 1/2 off the diagonal)")
for q in Q_GRID:
    ours = trivial_sheaf(DIGON_FREE, SMALL_N, q)
    reference = magnetic_laplacian(DIGON_FREE, SMALL_N, q)
    print(
        f"  q={q:<5} |L - 2 L^(q)| = "
        f"{float((ours - 2 * reference).abs().max()):.2e}"
        f"    |L - L^(q)| = {float((ours - reference).abs().max()):.2e}"
    )

print()
print("all-digon graph (i.e. undirected): the factor 2 disappears")
for q in (0.0, 0.25, 0.5):
    ours = trivial_sheaf(ALL_DIGON, SMALL_N, q)
    reference = magnetic_laplacian(ALL_DIGON, SMALL_N, q)
    print(
        f"  q={q:<5} |L - L^(q)| = "
        f"{float((ours - reference).abs().max()):.2e}"
    )

print()
sigma = sign_magnetic_laplacian(DIGON_FREE, SMALL_N)
print("Sign-Magnetic Laplacian at q = 1/4")
print(
    f"  |L^(1/4) - L^sigma| = "
    f"{float((magnetic_laplacian(DIGON_FREE, SMALL_N, 0.25) - sigma).abs().max()):.2e}"
)
print(
    f"  |ours - 2 L^sigma|  = "
    f"{float((trivial_sheaf(DIGON_FREE, SMALL_N, 0.25) - 2 * sigma).abs().max()):.2e}"
)

In [ ]:
# A 5-node graph, so individual entries are legible: the operator is 5x5 and
# every cell can carry its value.
q = 0.25
ours = trivial_sheaf(DIGON_FREE, SMALL_N, q) / 2
reference = magnetic_laplacian(DIGON_FREE, SMALL_N, q)
panels = [
    (r"$\Re$ (ours / 2)", ours.real),
    (r"$\Im$ (ours / 2)", ours.imag),
    (r"$\Re\ L^{(q)}$", reference.real),
    (r"$\Im\ L^{(q)}$", reference.imag),
]
limit = max(float(p[1].abs().max()) for p in panels) or 1.0
fig, axes = plt.subplots(1, 4, figsize=(10.0, 2.9), constrained_layout=True)
for ax, (title, matrix) in zip(axes, panels):
    ax.imshow(matrix.numpy(), cmap=SIGNED, vmin=-limit, vmax=limit)
    for i in range(SMALL_N):
        for j in range(SMALL_N):
            value = float(matrix[i, j])
            if abs(value) > 1e-12:
                ax.text(
                    j,
                    i,
                    f"{value:+.2f}",
                    ha="center",
                    va="center",
                    fontsize=6.5,
                    color=INK,
                )
    ax.set_title(title, fontsize=9.5)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
fig.suptitle(
    f"Theorem 4 at q = {q}: residual "
    f"{float((ours - reference).abs().max()):.1e}",
    fontsize=10.5,
)
save(fig, "magnetic_equivalence")
plt.show()

> **Expected output** &mdash; `media/magnetic_equivalence.png`
>
> ![Four panels showing the trivial-sheaf operator beside the Magnetic Laplacian, and their difference, confirming the two agree up to the factor of two that the paper's half-weight convention introduces.](media/magnetic_equivalence.png)

In [ ]:
# The sign argument in one figure: both conventions match the Magnetic
# Laplacian built with the SAME sign, so neither is singled out.
qs = np.linspace(0.0, 1.0, 81)
matched_plus, matched_minus, crossed = [], [], []
for q in qs:
    plus = trivial_sheaf(DIGON_FREE, SMALL_N, float(q), phase_sign=1)
    minus = trivial_sheaf(DIGON_FREE, SMALL_N, float(q), phase_sign=-1)
    ref_plus = 2 * magnetic_laplacian(DIGON_FREE, SMALL_N, float(q), sign=1)
    ref_minus = 2 * magnetic_laplacian(DIGON_FREE, SMALL_N, float(q), sign=-1)
    matched_plus.append(float((plus - ref_plus).abs().max()))
    matched_minus.append(float((minus - ref_minus).abs().max()))
    crossed.append(float((plus - ref_minus).abs().max()))

fig, ax = plt.subplots(figsize=(6.6, 3.6), constrained_layout=True)
floor = np.finfo(np.float64).eps
ax.semilogy(
    qs,
    np.maximum(matched_plus, floor / 4),
    color=SERIES[0],
    lw=2.0,
    label=r"phase_sign $=+1$ vs $\exp(+i\cdot)$",
)
ax.semilogy(
    qs,
    np.maximum(matched_minus, floor / 4),
    color=SERIES[2],
    lw=2.0,
    ls="--",
    label=r"phase_sign $=-1$ vs $\exp(-i\cdot)$",
)
ax.semilogy(
    qs,
    np.maximum(crossed, floor / 4),
    color=SERIES[1],
    lw=1.8,
    label=r"mismatched conventions",
)
ax.axhline(floor, color=MUTED, ls=":", lw=1.0)
ax.set_xlabel("charge $q$")
ax.set_ylabel(r"$\max|L - 2L^{(q)}|$")
ax.set_title("Either reading of Definition 1 satisfies Theorem 4")
ax.legend(loc="center right")
save(fig, "sign_convention")
plt.show()

print("The mismatched curve touches zero only at q = 0 and q = 1/2, where")
print(
    "exp(+i pi) = exp(-i pi), and is O(1) elsewhere. Both matched curves sit"
)
print("at machine epsilon for every q. Hence:")
for q in (0.1, 0.25, 0.4):
    plus = trivial_sheaf(DIGON_FREE, SMALL_N, q, phase_sign=1)
    minus = trivial_sheaf(DIGON_FREE, SMALL_N, q, phase_sign=-1)
    print(
        f"  q={q:<5} conjugate: "
        f"{torch.allclose(minus, plus.conj(), atol=1e-12)}"
        f"   isospectral: "
        f"{torch.allclose(torch.linalg.eigvalsh(plus), torch.linalg.eigvalsh(minus), atol=1e-10)}"
    )

> **Expected output** &mdash; `media/sign_convention.png`
>
> ![The residual against the Magnetic Laplacian for both readings of the sign in Definition 1, plotted over the charge grid. Both conventions satisfy Theorem 4 against their matching MagNet convention.](media/sign_convention.png)

**Read.** The two conventions are complex conjugates of one another, hence
isospectral, and the model absorbs the difference: the weights of Eq. 6 are
real, $\Im(X^{0}) = 0$, the nonlinearity gates on $\Re(z)$ only (so
$\overline{\sigma(z)} = \sigma(\bar z)$), and the readout is a learned real map
on $(\Re \Vert \Im)$ that can negate its second half. The same argument makes
$q$ and $1 - q$ equivalent, so the search range $[0, 1/2]$ suffices. We default
to the printed formula and expose the other as `phase_sign`.

## 5. Theorem 5: the complex incidence factorization

Theorem 5 gives the first incidence-matrix factorization of the Magnetic
Laplacian: with $\hat{B}_{ue}$ equal to $1$ at an arc's tail and
$-T^{(q)}_{uv}$ at its head, $\hat{B}\hat{B}^{*}$ is the operator.

**Claim.** $\hat{B}\hat{B}^{*}$ reproduces our trivial-sheaf operator exactly,
each column of $\hat B$ has exactly two non-zeros of unit modulus, and at
$q = 0$ the all-ones vector lies in the kernel of $\hat{B}^{*}$.

In [ ]:
def complex_incidence(edge_index, num_nodes, q, sign=1):
    """B_hat of Theorem 5."""
    adjacency = binary_adjacency(edge_index, num_nodes)
    representatives = []
    for u in range(num_nodes):
        for v in range(u + 1, num_nodes):
            forward, backward = adjacency[u, v] > 0, adjacency[v, u] > 0
            if not (forward or backward):
                continue
            representatives.append(
                (v, u) if backward and not forward else (u, v)
            )
    incidence = torch.zeros(
        num_nodes, len(representatives), dtype=torch.complex128
    )
    for column, (tail, head) in enumerate(representatives):
        # "-T_uv if e = (v,u)": at the head the phase is indexed the other way
        # round, i.e. conjugated.
        theta = torch.tensor(
            sign
            * 2
            * torch.pi
            * q
            * float(adjacency[head, tail] - adjacency[tail, head])
        )
        incidence[tail, column] = 1.0
        incidence[head, column] = -complex(torch.cos(theta), torch.sin(theta))
    return incidence


for q in (0.0, 0.25, 0.31, 0.5):
    incidence = complex_incidence(DIGON_FREE, SMALL_N, q)
    product = incidence @ incidence.conj().T
    ours = trivial_sheaf(DIGON_FREE, SMALL_N, q)
    print(
        f"  q={q:<5} |B B* - ours| = "
        f"{float((product - ours).abs().max()):.2e}"
        f"    |B B* - 2 L^(q)| = "
        f"{float((product - 2 * magnetic_laplacian(DIGON_FREE, SMALL_N, q)).abs().max()):.2e}"
    )

incidence = complex_incidence(DIGON_FREE, SMALL_N, 0.25)
magnitudes = incidence.abs()
print()
print(f"non-zeros per column : {(magnitudes > 0).sum(0).tolist()}")
print(
    f"all of unit modulus  : "
    f"{torch.allclose(magnitudes[magnitudes > 0], torch.ones(int((magnitudes > 0).sum()), dtype=DTYPE), atol=1e-12)}"
)
zero_charge = complex_incidence(DIGON_FREE, SMALL_N, 0.0)
kernel = zero_charge.conj().T @ torch.ones(SMALL_N, dtype=torch.complex128)
print(
    f"ones in ker(B*) at q=0: {torch.allclose(kernel, torch.zeros_like(kernel), atol=1e-12)}"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.4, 3.2), constrained_layout=True)

incidence = complex_incidence(DIGON_FREE, SMALL_N, 0.25)
# Hue carries the argument of each entry, so the phase is visible directly.
argument = np.angle(incidence.numpy())
mask = incidence.abs().numpy() > 0
shown = np.where(mask, argument, np.nan)
image = axes[0].imshow(shown, cmap="twilight", vmin=-np.pi, vmax=np.pi)
axes[0].set_title(r"$\arg \hat{B}$   (blank = zero)", fontsize=10)
axes[0].set_xlabel("edge")
axes[0].set_ylabel("node")
axes[0].set_xticks(range(incidence.size(1)))
axes[0].set_yticks(range(SMALL_N))
axes[0].grid(False)
bar = fig.colorbar(image, ax=axes[0], fraction=0.046)
bar.set_ticks([-np.pi, 0, np.pi])
bar.set_ticklabels([r"$-\pi$", "0", r"$\pi$"])

residual = (
    (incidence @ incidence.conj().T - trivial_sheaf(DIGON_FREE, SMALL_N, 0.25))
    .abs()
    .numpy()
)
floor = np.finfo(np.float64).eps
image = axes[1].imshow(
    np.log10(np.maximum(residual, floor / 16)), cmap="Greys_r"
)
axes[1].set_title(r"$\log_{10}|\hat{B}\hat{B}^{*} - L|$", fontsize=10)
axes[1].set_xticks(range(SMALL_N))
axes[1].set_yticks(range(SMALL_N))
axes[1].grid(False)
fig.colorbar(image, ax=axes[1], fraction=0.046)
save(fig, "incidence_factorization")
plt.show()

> **Expected output** &mdash; `media/incidence_factorization.png`
>
> ![Left, the argument of the complex incidence matrix, blank where entries are zero, showing exactly two non-zeros per column. Right, the base-ten logarithm of the residual between the factorization and the operator, uniformly below ten to the minus sixteen.](media/incidence_factorization.png)

**Read.** The residual is uniformly below $10^{-16}$. Note the columns with
argument $0$ and $\pi$: those are the reciprocal pairs, whose phase is exactly
$\pm 1$, sitting alongside the one-way arcs at $\pm\pi/2$. Getting this
factorization to hold pins the convention at the head of each arc — the phase
there is $T_{vu}$, not $T_{uv}$, and using the wrong one leaves an operator
that is still Hermitian and still positive semidefinite but no longer equal to
$L^{(q)}$ for any complex charge.

## 6. Why the edge pairing was re-implemented

Assembling the operator needs, for each undirected edge, the two restriction
maps at its endpoints — so each arc must be matched to its reverse. The Neural
Sheaf Diffusion port in TopoBench does that with an $[m/2, m, 2]$ broadcast
comparison, which is $O(m^2)$ in memory.

**Claim.** The sort-based pairing in `dsnn_utils.laplace` agrees with that
reference exactly wherever the reference can run, and scales where it cannot.
At the scale of the reproduction in section 7 the quadratic version would need
hundreds of gigabytes.

In [ ]:
from topobench.nn.backbones.graph.nsd_utils import laplace as nsd_laplace

# Agreement on a graph small enough for both.
support = laplace.symmetrize_support(MIXED, MIXED_N)
ours_lr, ours_pair = laplace.compute_left_right_map_index(support, MIXED_N)
ref_lr, ref_pair = nsd_laplace.compute_left_right_map_index(support)
print(f"pairing agrees with the reference: {torch.equal(ours_lr, ref_lr)}")
print(f"pairs agree                     : {torch.equal(ours_pair, ref_pair)}")

sizes, ours_time, ref_time = [2000, 4000, 8000, 16000], [], []
for arcs in sizes:
    generator = torch.Generator().manual_seed(0)
    num_nodes = arcs // 2
    src = torch.randint(0, num_nodes, (arcs,), generator=generator)
    dst = torch.randint(0, num_nodes, (arcs,), generator=generator)
    graph = laplace.symmetrize_support(torch.stack([src, dst]), num_nodes)
    start = time.perf_counter()
    laplace.compute_left_right_map_index(graph, num_nodes)
    ours_time.append(time.perf_counter() - start)
    start = time.perf_counter()
    nsd_laplace.compute_left_right_map_index(graph)
    ref_time.append(time.perf_counter() - start)
    sizes[sizes.index(arcs)] = graph.size(1)

slope_ours = np.polyfit(np.log(sizes), np.log(np.maximum(ours_time, 1e-6)), 1)[
    0
]
slope_ref = np.polyfit(np.log(sizes), np.log(ref_time), 1)[0]
print()
print(f"fitted log-log slope, sorting  : {slope_ours:.2f}")
print(f"fitted log-log slope, broadcast: {slope_ref:.2f}")

# The reproduction of section 7 runs at this many arcs.
DSBM_ARCS = 263_052
needed = DSBM_ARCS * (DSBM_ARCS // 2) * 2
print()
print(
    f"at the section 7 scale ({DSBM_ARCS:,} arcs) the broadcast comparison"
    f"\nwould allocate about {needed / 1024 ** 3:,.0f} GiB of booleans alone."
)

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 3.6), constrained_layout=True)
ax.loglog(
    sizes,
    ours_time,
    marker="o",
    color=SERIES[0],
    lw=2.0,
    label=f"sort-based (slope {slope_ours:.2f})",
)
ax.loglog(
    sizes,
    ref_time,
    marker="s",
    color=SERIES[1],
    lw=2.0,
    label=f"broadcast comparison (slope {slope_ref:.2f})",
)
ax.set_xlabel("arcs in the symmetrized support")
ax.set_ylabel("seconds")
ax.set_title("Edge pairing: measured scaling")
ax.legend()
save(fig, "index_scaling")
plt.show()

> **Expected output** &mdash; `media/index_scaling.png`
>
> ![Runtime of the sort-based edge pairing against the quadratic reference as the arc count grows, on log axes, with measured slopes close to one and two respectively.](media/index_scaling.png)

**Read.** The measured slopes are close to 1 and 2 respectively. We
deliberately stop at 16k arcs, well before the reference would exhaust memory,
since the slope is what the comparison rests on. This is also why section 7 uses DSNN at
$q = 0$ as its direction-blind baseline instead of the `NSDEncoder` already in
the repository: Theorem 3 proves the two operators are the same object, section
3 verifies it bit-for-bit, and only one of them can be assembled at $n = 2500$.

We left `nsd_utils` untouched; the replacement lives in `dsnn_utils` and also
tolerates self-loops, which trip the reference's edge-count assertion.

## 7. The paper's Figure 2, reproduced

This is the experiment the paper uses to isolate its contribution, and the one
the challenge benchmark cannot express.

**The generator** (Appendix E). Sample an undirected graph on $n$ nodes split
into $C$ equal communities, joining $u \in C_i$ to $v \in C_j$ with probability
$\alpha_{ii} = 0.1$ within a community and $\alpha_{ij}$ across. Then orient
every edge: for $u \in C_i, v \in C_j$ send $u \to v$ with probability
$\beta_{ij}$, where $\beta_{ij} + \beta_{ji} = 1$ and $\beta = 0.2$. Node
features are **one-dimensional**: in-degree plus out-degree.

That feature is invariant to the orientation, so it carries no class signal at
all, and the undirected skeleton is a homogeneous block model. **The community
label is recoverable only from the direction of the edges.** A direction-blind
model is therefore pinned at chance, $1/C = 20\%$, by construction — we verify
that with a probe rather than by retraining eight baselines.

**Claim.** DSNN with a non-zero charge reaches the paper's 86–96% band, while
the same model at $q = 0$ — which Theorem 3 makes exactly direction-blind sheaf
diffusion — stays at chance.

In [ ]:
FULL_PROTOCOL = True  # set False for a fast pass (n = 800, one seed)

if FULL_PROTOCOL:
    DSBM_N, SEEDS, EPOCHS = 2500, (0, 1), 80
else:
    DSBM_N, SEEDS, EPOCHS = 800, (0,), 60

ALPHAS = (0.05, 0.08, 0.10)
CHANCE = 1 / 5

print("Protocol, and exactly how it is reduced from the paper:")
print(
    f"  {'generator (n, C, alpha_ii, alpha_ij, beta, features, splits)':60s} unchanged"
)
print(f"  {'n':60s} {DSBM_N} (paper: 2500)")
print(f"  {'seeds':60s} {len(SEEDS)} (paper: 10)")
print(f"  {'epochs / patience':60s} {EPOCHS} / none (paper: 1000 / 200)")
print(
    f"  {'hyperparameter search':60s} none; one fixed setting for every model"
)
print(
    f"  {'baselines':60s} DSNN at q=0 (= direction-blind, Thm 3) + a feature probe"
)
print()
print(
    "Every model below uses d=2, hidden=32, 2 layers, no dropout, Adam lr=1e-2,"
)
print("all of which are points of the paper's grid.")

In [ ]:
def dsbm(
    n, n_communities=5, alpha_intra=0.10, alpha_inter=0.08, beta=0.20, seed=0
):
    """The directed stochastic block model of Appendix E.

    Returns the directed edge index, the 1-D degree feature, and the labels.
    """
    generator = torch.Generator().manual_seed(seed)
    labels = (torch.arange(n) * n_communities) // n
    same = labels.view(-1, 1) == labels.view(1, -1)
    probability = torch.where(
        same, torch.tensor(alpha_intra), torch.tensor(alpha_inter)
    )
    upper = torch.triu(
        torch.rand(n, n, generator=generator) < probability, diagonal=1
    )
    u, v = upper.nonzero(as_tuple=True)
    # beta_ij + beta_ji = 1, so within a community the coin is necessarily fair.
    forward_probability = torch.where(
        labels[u] < labels[v],
        torch.tensor(beta),
        torch.where(
            labels[u] > labels[v], torch.tensor(1 - beta), torch.tensor(0.5)
        ),
    )
    forward = torch.rand(u.numel(), generator=generator) < forward_probability
    edge_index = torch.stack(
        [torch.where(forward, u, v), torch.where(forward, v, u)]
    )
    degree = torch.bincount(edge_index[0], minlength=n) + torch.bincount(
        edge_index[1], minlength=n
    )
    return edge_index, degree.to(torch.float32).unsqueeze(1), labels


def dsbm_splits(labels, seed=0):
    """80 / 5 / 15 node split, as in Appendix E."""
    generator = torch.Generator().manual_seed(seed)
    order = torch.randperm(labels.numel(), generator=generator)
    n_train = int(0.80 * labels.numel())
    n_val = int(0.05 * labels.numel())
    return (
        order[:n_train],
        order[n_train : n_train + n_val],
        order[n_train + n_val :],
    )


def train_dsnn(
    edge_index, features, labels, masks, *, q, family="diag", epochs=80, seed=0
):
    """Train the raw backbone with a linear head; return held-out accuracy."""
    torch.manual_seed(seed)
    train_mask, _, test_mask = masks
    model = DSNNEncoder(
        input_dim=features.size(1),
        hidden_dim=32,
        num_layers=2,
        d=2,
        q=q,
        sheaf_type=family,
        dropout=0.0,
        input_dropout=0.0,
    )
    head = torch.nn.Linear(32, int(labels.max()) + 1)
    optimiser = torch.optim.Adam(
        list(model.parameters()) + list(head.parameters()), lr=1e-2
    )
    start = time.perf_counter()
    for _ in range(epochs):
        model.train()
        optimiser.zero_grad()
        logits = head(model(features, edge_index))
        torch.nn.functional.cross_entropy(
            logits[train_mask], labels[train_mask]
        ).backward()
        optimiser.step()
    model.eval()
    with torch.no_grad():
        predicted = head(model(features, edge_index)).argmax(1)
    accuracy = (
        (predicted[test_mask] == labels[test_mask]).double().mean().item()
    )
    return accuracy, (time.perf_counter() - start) / epochs


edge_index, features, labels = dsbm(DSBM_N, alpha_inter=0.08, seed=0)
print(
    f"n={DSBM_N}  arcs={edge_index.size(1):,}  mean degree="
    f"{float(features.mean()):.1f}  communities={int(labels.max()) + 1}"
)

In [ ]:
# The feature probe: can anything be read off the 1-D degree feature alone?
from sklearn.linear_model import LogisticRegression

probe_scores = []
for seed in SEEDS:
    edge_index, features, labels = dsbm(DSBM_N, alpha_inter=0.08, seed=seed)
    train_mask, _, test_mask = dsbm_splits(labels, seed=seed)
    probe = LogisticRegression(max_iter=1000).fit(
        features[train_mask].numpy(), labels[train_mask].numpy()
    )
    probe_scores.append(
        probe.score(features[test_mask].numpy(), labels[test_mask].numpy())
    )
print(
    f"logistic regression on the degree feature alone: "
    f"{np.mean(probe_scores):.3f} +- {np.std(probe_scores):.3f}"
)
print(f"chance for C = 5                              : {CHANCE:.3f}")
print()
print(
    "In-degree plus out-degree is invariant to the orientation, so this is not"
    "\na weak baseline, it is the strongest any direction-blind feature model"
    "\ncan be. That is why the paper's other baselines all sit at chance."
)

In [ ]:
# Cached so the figure can be redrawn without repeating the run. Delete
# 2026_tdl_challenge/dsbm_results.json to force a retrain (~45 min on CPU).
import json

DSBM_CACHE = REPO / "2026_tdl_challenge" / "dsbm_results.json"

if DSBM_CACHE.exists():
    cached = json.loads(DSBM_CACHE.read_text())
    results = {(r["model"], r["alpha"]): r["scores"] for r in cached["runs"]}
    print(f"loaded {len(results)} settings from {DSBM_CACHE.name}")
else:
    results = {}
    total = len(ALPHAS) * len(SEEDS) * 2 + 2 * len(SEEDS)
    done = 0
    print(f"running {total} trainings\n")

    for alpha in ALPHAS:
        for q, tag in ((0.25, "Diag-DSNN, q=0.25"), (0.0, "DSNN q=0 (= NSD)")):
            scores = []
            for seed in SEEDS:
                edge_index, features, labels = dsbm(
                    DSBM_N, alpha_inter=alpha, seed=seed
                )
                masks = dsbm_splits(labels, seed=seed)
                accuracy, per_epoch = train_dsnn(
                    edge_index,
                    features,
                    labels,
                    masks,
                    q=q,
                    epochs=EPOCHS,
                    seed=seed,
                )
                scores.append(accuracy)
                done += 1
                print(
                    f"  [{done:2d}/{total}] alpha={alpha:.2f} {tag:20s} "
                    f"seed={seed}  acc={accuracy:.3f}  ({per_epoch:.2f} s/epoch)"
                )
            results[(tag, alpha)] = scores

    # The other two families, at the hardest setting only.
    for family, tag in (
        ("bundle", "O(d)-DSNN, q=0.25"),
        ("general", "Gen-DSNN, q=0.25"),
    ):
        scores = []
        for seed in SEEDS:
            edge_index, features, labels = dsbm(
                DSBM_N, alpha_inter=0.10, seed=seed
            )
            masks = dsbm_splits(labels, seed=seed)
            accuracy, per_epoch = train_dsnn(
                edge_index,
                features,
                labels,
                masks,
                q=0.25,
                family=family,
                epochs=EPOCHS,
                seed=seed,
            )
            scores.append(accuracy)
            done += 1
            print(
                f"  [{done:2d}/{total}] alpha=0.10 {tag:20s} "
                f"seed={seed}  acc={accuracy:.3f}  ({per_epoch:.2f} s/epoch)"
            )
        results[(tag, 0.10)] = scores

    DSBM_CACHE.write_text(
        json.dumps(
            {
                "protocol": {
                    "n": DSBM_N,
                    "communities": 5,
                    "seeds": list(SEEDS),
                    "epochs": EPOCHS,
                },
                "runs": [
                    {"model": k[0], "alpha": k[1], "scores": v}
                    for k, v in results.items()
                ],
            },
            indent=2,
        )
        + "\n"
    )

print("\nsummary (mean +- std over seeds)")
for key in sorted(results, key=lambda k: (k[0], k[1])):
    scores = results[key]
    print(
        f"  {key[0]:20s} alpha={key[1]:.2f}  "
        f"{np.mean(scores):.3f} +- {np.std(scores):.3f}"
    )


In [ ]:
def annotate(axis, bars, means):
    for bar, mean in zip(bars, means):
        axis.annotate(
            f"{mean:.2f}",
            xy=(bar.get_x() + bar.get_width() / 2, mean),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            fontsize=8.5,
            color=INK,
        )


fig, (ax, ax2) = plt.subplots(
    1,
    2,
    figsize=(9.2, 4.0),
    width_ratios=[3.0, 1.0],
    sharey=True,
    constrained_layout=True,
)

labels = ("Diag-DSNN, $q = 0.25$", "same model, $q = 0$")
tags = ("Diag-DSNN, q=0.25", "DSNN q=0 (= NSD)")
width = 0.34
positions = np.arange(len(ALPHAS))
for k, (label, tag) in enumerate(zip(labels, tags)):
    means = [np.mean(results[(tag, a)]) for a in ALPHAS]
    errors = [np.std(results[(tag, a)]) for a in ALPHAS]
    bars = ax.bar(
        positions + (k - 0.5) * width,
        means,
        width,
        yerr=errors,
        capsize=3,
        color=SERIES[k],
        label=label,
        zorder=3,
    )
    # Only the upper series is annotated; the lower one sits on the chance
    # line, which already says where it is.
    if k == 0:
        annotate(ax, bars, means)

ax.axhspan(
    0.86, 0.96, color=SERIES[0], alpha=0.10, zorder=0,
    label="paper's band, 86-96%",
)
ax.axhline(
    CHANCE, color=INK, ls="--", lw=1.2, zorder=2,
    label=f"chance, $1/C$ = {CHANCE:.0%}",
)
ax.set_xticks(positions)
ax.set_xticklabels([rf"$\alpha_{{ij}} = {a}$" for a in ALPHAS])
ax.set_ylabel("held-out accuracy")
ax.set_ylim(0, 1.08)
# The gap between the two series is empty at every x, so the legend sits
# there without covering a bar or a value label.
ax.legend(loc="center left", bbox_to_anchor=(0.02, 0.52), framealpha=0.0)

others = ("O(d)-DSNN, q=0.25", "Gen-DSNN, q=0.25")
means = [np.mean(results[(t, 0.10)]) for t in others]
errors = [np.std(results[(t, 0.10)]) for t in others]
bars = ax2.bar(
    range(len(others)), means, 0.55, yerr=errors, capsize=3,
    color=[SERIES[2], SERIES[3]], zorder=3,
)
annotate(ax2, bars, means)
ax2.axhline(CHANCE, color=INK, ls="--", lw=1.2, zorder=2)
ax2.set_xticks(range(len(others)))
ax2.set_xticklabels(["O(d)", "Gen"])
ax2.set_title(r"other families, $\alpha_{ij} = 0.10$", fontsize=10)

fig.suptitle(
    f"Figure 2 reproduced: n = {DSBM_N}, C = 5, {len(SEEDS)} seeds, "
    f"{EPOCHS} epochs",
    fontsize=11.5,
    fontweight="600",
)
save(fig, "dsbm_figure2")
plt.show()


> **Expected output** &mdash; `media/dsbm_figure2.png`
>
> ![Held-out accuracy on the directed stochastic block model against inter-community density. DSNN with a non-zero charge lands in the paper's band while the same model at q equals zero, and a feature-only probe, sit at chance. An inset repeats the comparison for the orthogonal and general families.](media/dsbm_figure2.png)

**Read.** With the charge on, the diagonal family
lands in or beside the paper's 86–96% band; with the charge off it sits at
chance. The two runs differ *only* in $q$, so the gap is attributable to the
phase and to nothing else. At the hardest inter-community density the
orthogonal and general families both land in the same band as the diagonal
one.

The $q = 0$ bar is a strong baseline: by Theorem 3 it is exactly the real
sheaf Laplacian, i.e. Neural Sheaf Diffusion with this readout, and section 3
verified that identity bit-for-bit. Together with the feature probe it brackets
what any direction-blind model can do here.

Where our numbers sit slightly below the paper's, the reductions listed above
are the likely cause — fewer seeds, far fewer epochs, and no hyperparameter
search at all. We did not tune to close the gap.

## 8. Does an induced orientation change what the model learns?

Sections 3 and 7 are exact statements about the operator. This is a separate,
empirical question, and a narrower one than "does direction help": we compare
`orientation` settings on undirected input, so what is isolated is the phase
that an induced orientation manufactures, not directionality in data.

**Claim.** Accuracy must be *identical* in the three cells where the operator
provably cannot depend on the charge — `none` at any $q$, and any orientation
at $q = 0$ — and may differ only where an orientation and a non-zero charge are
both present. On a degree-regular graph even `degree` must coincide with
`none`.

In [ ]:
def hierarchy_task(n=240, levels=4, p_forward=0.10, seed=0):
    """An undirected graph whose label is a latent level, plus noise edges."""
    generator = torch.Generator().manual_seed(seed)
    labels = (torch.arange(n) * levels) // n
    src, dst = [], []
    for u in range(n):
        for v in range(n):
            if u >= v:
                continue
            gap = int(labels[v]) - int(labels[u])
            probability = p_forward if gap == 1 else 0.02 * (gap == 0)
            if (
                probability
                and torch.rand(1, generator=generator).item() < probability
            ):
                src += [u, v]
                dst += [v, u]
    edge_index = torch.tensor([src, dst])
    degree = torch.bincount(edge_index[0], minlength=n).to(torch.float32)
    features = torch.stack([degree, torch.ones(n)], dim=1)
    return edge_index, features, labels


def run_cell(edge_index, features, labels, orientation, q, seed):
    """Train once and report held-out accuracy."""
    torch.manual_seed(seed)
    n = features.size(0)
    order = torch.randperm(n, generator=torch.Generator().manual_seed(seed))
    train_mask, test_mask = order[: n // 2], order[n // 2 :]
    model = DSNNEncoder(
        input_dim=features.size(1),
        hidden_dim=16,
        num_layers=2,
        d=2,
        q=q,
        orientation=orientation,
        dropout=0.0,
        input_dropout=0.0,
    )
    head = torch.nn.Linear(16, int(labels.max()) + 1)
    optimiser = torch.optim.Adam(
        list(model.parameters()) + list(head.parameters()), lr=0.02
    )
    for _ in range(150):
        model.train()
        optimiser.zero_grad()
        logits = head(model(features, edge_index))
        torch.nn.functional.cross_entropy(
            logits[train_mask], labels[train_mask]
        ).backward()
        optimiser.step()
    model.eval()
    with torch.no_grad():
        predicted = head(model(features, edge_index)).argmax(1)
    return (predicted[test_mask] == labels[test_mask]).double().mean().item()


edge_index, features, labels = hierarchy_task()
print(
    f"nodes={features.size(0)} arcs={edge_index.size(1)} "
    f"undirected={bool(is_undirected(edge_index))}"
)

CELLS_SWEEP = [
    ("none", 0.0),
    ("none", 0.25),
    ("degree", 0.0),
    ("degree", 0.25),
    ("index", 0.0),
    ("index", 0.25),
]
sweep = {}
for orientation, q in CELLS_SWEEP:
    scores = [
        run_cell(edge_index, features, labels, orientation, q, seed)
        for seed in range(5)
    ]
    sweep[(orientation, q)] = scores
    print(
        f"  orientation={orientation:7s} q={q:<5} "
        f"{np.mean(scores):.3f} +- {np.std(scores):.3f}"
    )

print()
print("invariances that must hold exactly:")
print(
    f"  none:   q=0 vs q=0.25   identical: "
    f"{sweep[('none', 0.0)] == sweep[('none', 0.25)]}"
)
print(
    f"  degree: q=0 vs none q=0 identical: "
    f"{sweep[('degree', 0.0)] == sweep[('none', 0.0)]}"
)
print(
    f"  index:  q=0 vs none q=0 identical: "
    f"{sweep[('index', 0.0)] == sweep[('none', 0.0)]}"
)

In [ ]:
# The regular-graph corner: an equivariant degree orientation cannot orient a
# graph in which every node looks the same.
cycle = torch.tensor(
    [
        [i for i in range(20)] + [(i + 1) % 20 for i in range(20)],
        [(i + 1) % 20 for i in range(20)] + [i for i in range(20)],
    ]
)
for orientation in ("none", "degree", "index"):
    support = laplace.symmetrize_support(cycle, 20)
    _, pair = laplace.compute_left_right_map_index(support, 20)
    sign = phase.pair_sign(cycle, support, pair, 20, orientation)
    print(
        f"  20-cycle, orientation={orientation:7s} oriented "
        f"{int((sign != 0).sum())}/{sign.numel()} edges"
    )
print()
print("On a degree-regular graph 'degree' orients nothing and the model is")
print("bit-identical to 'none'. That is the cost of permutation equivariance:")
print("breaking ties by node index would orient everything, at the price of")
print("making the operator depend on the node ordering.")

fig, ax = plt.subplots(figsize=(7.0, 3.8), constrained_layout=True)
orientations = ("none", "degree", "index")
width = 0.36
positions = np.arange(len(orientations))
for k, q in enumerate((0.0, 0.25)):
    means = [np.mean(sweep[(o, q)]) for o in orientations]
    errors = [np.std(sweep[(o, q)]) for o in orientations]
    bars = ax.bar(
        positions + (k - 0.5) * width,
        means,
        width,
        yerr=errors,
        capsize=3,
        color=SERIES[k],
        label=f"q = {q}",
    )
    for bar, mean in zip(bars, means):
        ax.annotate(
            f"{mean:.2f}",
            xy=(bar.get_x() + bar.get_width() / 2, mean),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center",
            fontsize=8.5,
            color=INK,
        )
ax.set_xticks(positions)
ax.set_xticklabels(orientations)
ax.set_xlabel("orientation")
ax.set_ylabel("held-out accuracy")
ax.set_title(
    "Undirected input: the charge only matters once a direction exists"
)
ax.legend()
save(fig, "held_out_accuracy")
plt.show()

> **Expected output** &mdash; `media/held_out_accuracy.png`
>
> ![Held-out accuracy for each orientation and charge on undirected input. The three cells where the operator provably cannot depend on the charge give identical bars; only an orientation combined with a non-zero charge moves the result.](media/held_out_accuracy.png)

**Read.** The three invariance checks are exact equalities rather than
approximate ones, which rules out the code using the charge anywhere the
mathematics says there is none to use. Whether the manufactured phase *helps* is a smaller and
noisier question, five seeds on one synthetic task; treat those magnitudes as
indicative and the invariances as exact.

There is a structural reason not to expect much from an induced orientation on
sparse graphs. A phase derived from a graph distance is a pure gauge
transformation, $L = U^{*}L^{\mathcal{F}}U$, hence isospectral with the real
operator; and on a tree *every* orientation is such a gauge. What makes a phase
genuinely directional is non-zero holonomy around cycles. GraphUniverse runs at
average degree 1–2 in half the grid, which is close to tree-like, so we would
not expect `dsnn_degree` to gain much there.

## 9. Summary

| claim | where in the paper | status |
|---|---|---|
| $L$ is Hermitian, diagonal exactly real | Thm 1, Eq. 2-3 | exact-verified, all 3 families |
| $L \succeq 0$ | Thm 1 | exact-verified (quadratic form and spectrum) |
| $\mathrm{spec}(L_N) \subseteq [0,2]$ | Thm 2, Eq. 5 | exact-verified across the whole $q$ grid; $\lambda_{\max} = 2.000000$ |
| Undirected input $\Rightarrow$ real operator, $q$ inert | Thm 3 | exact-verified: bitwise identical for all $q$ |
| Equals the real sheaf Laplacian of the NSD port | Thm 3 | exact-verified for the diagonal and general families |
| Trivial sheaf $=$ Magnetic Laplacian ($\times 2$ if digon-free) | Thm 4 | exact-verified over the $q$ grid |
| At $q=1/4$, $=$ Sign-Magnetic Laplacian | Thm 4 | exact-verified |
| $L = \hat{B}\hat{B}^{*}$ | Thm 5 | exact-verified |
| Definition 1's printed sign vs its example | Def. 1 | resolved: conjugate, isospectral, same model class |
| DSNN separates the DSBM, direction-blind models do not | Fig. 2, App. E | reproduced at $n=2500$ under a reduced protocol |
| Sort-based edge pairing matches the quadratic reference | ours | verified, with measured slopes ~1 vs ~2 |

Every GraphUniverse graph is undirected, so on the challenge grid the faithful
config is a real-valued model and the charge does nothing. We therefore submit
both halves:

* **`graph/dsnn`** — the faithful config, inventing no direction.
* **`graph/dsnn_degree`** — an induced, permutation-equivariant orientation, the
  only shipped config whose complex path is live on this benchmark, and the one
  the ranked `results.json` comes from.

The `results.json` committed under `2026_tdl_challenge/outputs/` is the one
`graph/dsnn_degree` produced.